# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [13]:
import numpy as np 
import pandas as pd

In [14]:
print(np.__version__)
print(pd.__version__)

2.4.6
3.0.2


In [15]:
# sanity checks
arr = np.array([1,2,3])
df = pd.DataFrame({"a":[1,2], "b":["x","y"]})
print(arr.sum(), df.head().to_dict())

6 {'a': {0: 1, 1: 2}, 'b': {0: 'x', 1: 'y'}}


## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use. 

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text. 

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available. 

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [16]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

#news = pd.read_csv('/kaggle/input/bbc-news/bbc_news.csv')
#MAX_NEWS = 1000
#DOCUMENT="description"
#TOPIC="title"

news = pd.read_csv(r"C:\Users\con2m\Desktop\IRONHACKCOURSE\Week 18\lab-langchain-med-main\lab-langchain-med-main\datasets\articles.csv", encoding="utf-8", low_memory=False)
MAX_NEWS = 100
DOCUMENT = "Article Body"
TOPIC = "Article Header"


#news = "PICK A DATASET" #Ideally pick one from the commented ones above

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [17]:
subset_news = news.head(MAX_NEWS).copy()
subset_news.reset_index(drop=True, inplace=True)
subset_news["id"] = subset_news.index.astype(str)

# Basic validations
if not subset_news["id"].is_unique:
    raise ValueError("IDs are not unique in subset_news")
if DOCUMENT not in subset_news.columns:
    raise KeyError(f"DOCUMENT column not found: {DOCUMENT}")
if TOPIC not in subset_news.columns:
    raise KeyError(f"TOPIC column not found: {TOPIC}")

print("subset shape:", subset_news.shape)
print("sample ids:", subset_news["id"].head().tolist())
print(subset_news[[DOCUMENT, TOPIC, "id"]].head(3))


subset shape: (100, 9)
sample ids: ['0', '1', '2', '3', '4']
                                        Article Body  \
0  ['Discovering new materials and drugs typicall...   
1  ['Is it possible to build machine-learning mod...   
2  ['From cameras to self-driving cars, many of t...   

                                      Article Header id  
0  Learning the language of molecules to predict ...  0  
1  MIT scientists build a system that can generat...  1  
2  When computer vision works more like a brain, ...  2  


In [18]:
# Verify and prepare lists for ChromaDB
# Assumes subset_news exists and DOCUMENT, TOPIC are defined

# 1. Ensure id column exists
if "id" not in subset_news.columns:
    raise KeyError("Missing 'id' column in subset_news")

# 2. Force string type and check nulls
subset_news["id"] = subset_news["id"].astype(str)
if subset_news["id"].isnull().any():
    raise ValueError("Null values found in 'id' column")

# 3. Ensure uniqueness; if not unique, recreate safe ids
if not subset_news["id"].is_unique:
    subset_news = subset_news.reset_index(drop=True).copy()
    subset_news["id"] = subset_news.index.astype(str)

# 4. Prepare lists for Chroma
documents = subset_news[DOCUMENT].astype(str).tolist()
ids = subset_news["id"].tolist()
metadatas = subset_news[[TOPIC]].fillna("").to_dict(orient="records")

# 5. Final sanity checks
assert len(documents) == len(ids) == len(metadatas), "Length mismatch"
print("Ready for Chroma:", len(ids), "documents")


Ready for Chroma: 100 documents


## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database. 

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior. 

In [12]:
# Install (use %pip in notebooks)
%pip install --upgrade chromadb

# Optional: install extras if you plan to run the server or use specific backends
%pip install --upgrade chromadb[server]


  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached websocket_client-1.9.0-py3-none-a

In [1]:
import importlib, sys

# Check that the package is importable and print version
try:
    chromadb = importlib.import_module("chromadb")
    print("chromadb imported:", chromadb.__version__)
except Exception as e:
    print("chromadb import error:", type(e).__name__, e)
    raise

# Check submodule import
try:
    from chromadb.config import Settings
    print("chromadb.config.Settings import OK")
except Exception as e:
    print("chromadb.config import error:", type(e).__name__, e)
    raise


chromadb imported: 1.5.9
chromadb.config.Settings import OK


In [3]:
import chromadb
import sys

print("chromadb version:", chromadb.__version__)
print("python executable:", sys.executable)

# Use the default client (no legacy Settings)
client = chromadb.Client()
print("Chroma client created (default):", type(client))

# Create a collection and add a tiny test doc to verify everything works
collection = client.create_collection(name="test_collection")
collection.add(
    ids=["t0"],
    documents=["Test document for Chroma sanity check."],
    metadatas=[{"title":"sanity"}]
)
print("Collection count:", collection.count())


chromadb version: 1.5.9
python executable: c:\Users\con2m\anaconda3\envs\mlops-env\python.exe
Chroma client created (default): <class 'chromadb.api.client.Client'>


C:\Users\con2m\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 23.7MiB/s]


Collection count: 1


In [5]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters 
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance. 

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that. 

In [19]:
chroma_client = chromadb.PersistentClient(
    path="C:/Users/con2m/Desktop/IRONHACKCOURSE/Week 18/lab-langchain-med-main/lab-langchain-med-main/chroma_db"
)


## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it. 

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [20]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
    

It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***. 
* In the **document** we store the big text, it's a different column in each Dataset. 
* In **metadatas**, we can informa a list of topics. 
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS. 


In [21]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [22]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

{'ids': [['id9', 'id85', 'id1', 'id80', 'id52', 'id27', 'id73', 'id93', 'id6', 'id98']], 'embeddings': None, 'documents': [['[\'How will advances in computing transform human society?\', \'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.\', \'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.\', \'Students dreamed u

## Vector MAP

In [24]:
%pip install scikit-learn


  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 56.8 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ----------------- ---------------------- 16.0/36.6 MB 71.8 MB/s eta 0:00:01
   ------------------------------- -------- 29.1/36.6 MB 68.3 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 61.2 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------

In [25]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [28]:

getado = collection.get(ids="id98", 
                       include=["documents", "embeddings"])


In [29]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([[ 1.64806414e-02, -1.05098225e-01,  4.12227660e-02,
        -5.36262617e-02, -3.33807361e-03,  1.62998363e-02,
        -1.88525883e-03, -2.60295644e-02,  1.11483270e-02,
        -1.75798219e-02, -7.94970840e-02, -5.01477979e-02,
        -1.61883906e-02,  4.76560602e-03,  1.39789563e-02,
         1.71011966e-03,  1.04735367e-01,  7.47525245e-02,
        -3.85921933e-02, -8.72070640e-02,  3.61388102e-02,
         2.71336604e-02,  3.50330286e-02, -5.48494747e-04,
        -1.01981256e-02,  3.87011021e-02,  3.91495004e-02,
        -6.17150143e-02, -1.51279941e-03, -2.74983281e-03,
         5.39355315e-02,  2.70635076e-02,  7.59760365e-02,
        -3.14551964e-02, -8.66764560e-02, -5.79601433e-03,
         5.92005216e-02, -9.58839338e-03,  2.63297390e-02,
        -2.09694616e-02, -7.04626292e-02, -3.14992331e-02,
         2.51802597e-02, -4.99141291e-02,  4.08012941e-02,
        -2.77277175e-02, -4.69609760e-02, -8.68649855e-02,
        -4.94138859e-02,  3.25066857e-02, -9.72862914e-0

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents. 

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search. 


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models. 

We are importing: 
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification. 

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2. 

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)




config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The next step is to initialize the pipeline using the objects created above. 

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [32]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user. 

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**. 

We only need to join the two parts together to create the prompt that we are going to send to the model. 

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part. 

In [34]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
context = context[:4096]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

'Relevant context: #[\'How will advances in computing transform human society?\', \'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.\', \'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.\', \'Students dreamed up highly inventive scenarios for how the technologies of today and tomorrow could impact society, for bet

Now all that remains is to send the prompt to the model and wait for its response!


In [35]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Relevant context: #['How will advances in computing transform human society?', 'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.', 'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.', 'Students dreamed up highly inventive scenarios for how the technologies of today and tomorrow could impact society, for better or w

In [37]:
# List of 10 dataset‑relevant questions
questions = [
    "What is the Envisioning the Future of Computing Prize at MIT?",
    "Who won the Envisioning the Future of Computing Prize and what was their essay about?",
    "What themes did MIT students explore in their essays about the future of computing?",
    "What is BioAutoMATED and what problem does it aim to solve?",
    "How does the LiGO method accelerate the training of large language models?",
    "What risks are associated with biased AI models in healthcare decision-making?",
    "How did MIT researchers study the impact of AI bias on crisis hotline decisions?",
    "What are the main advantages of using AutoML tools for biological sequence analysis?",
    "What concerns did MIT researchers raise about the societal impact of advanced computing technologies?",
    "How can descriptive AI recommendations reduce the harmful effects of biased models?"
]

# Build the context from the retrieved documents
context = " ".join([str(i) for i in results["documents"][0]])

# Limit context length to avoid memory issues
context = context[:4096]

# Loop through each question and generate a prompt + response
for q in questions:
    prompt_template = f"""
You are an assistant that answers ONLY using the provided context.
If the answer is not in the context, reply exactly with:
"The context does not contain the answer."

CONTEXT:
{context}

QUESTION:
{q}

ANSWER:
"""
    lm_response = pipe(prompt_template)
    print(f"\nQUESTION: {q}")
    print("ANSWER:", lm_response[0]["generated_text"])
    print("-" * 80)



QUESTION: What is the Envisioning the Future of Computing Prize at MIT?
ANSWER: 
You are an assistant that answers ONLY using the provided context.
If the answer is not in the context, reply exactly with:
"The context does not contain the answer."

CONTEXT:
['How will advances in computing transform human society?', 'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.', 'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive s

The final code block creates a list of 10 questions that are actually relevant to the dataset, builds a context string from the documents retrieved from the vector database, and then loops through each question to generate a RAG‑style prompt.

For every question, the code:

Constructs a strict prompt that forces the model to answer only using the provided context.

Prevents hallucinations by instructing the model to reply with
"The context does not contain the answer."  
if the information is missing.

Feeds the prompt into the text‑generation pipeline.

Prints both the question and the model’s answer.

This allows you to test how well your RAG system works with real, meaningful questions that match the content of your dataset.

🎯 Why we did this
We created this code for three important reasons:

1. To evaluate the RAG system properly
Using irrelevant questions (like “Can I buy a Toshiba laptop?”) makes the RAG look broken.
Using dataset‑aligned questions shows whether the system is actually working.

2. To enforce context‑based answers
The strict prompt prevents the model from inventing information and ensures it only uses what is retrieved from the vector database.

3. To test multiple questions automatically
Looping through 10 questions lets you quickly verify:

retrieval quality

prompt quality

model behavior

hallucination control

## ⭐ Full Lab Summary — End‑to‑End RAG Pipeline with Transformers, Chroma, and a Local LLM
#### 1. Environment Setup and Imports
We began by importing the core components required to build a Retrieval‑Augmented Generation (RAG) system using Hugging Face Transformers:

AutoTokenizer – loads the correct tokenizer for any model.

AutoModelForCausalLM – loads a causal language model for text generation.

pipeline – creates a high‑level text‑generation interface.

These imports form the foundation for loading and interacting with any modern LLM.

#### 2. Model Selection and Updates
Originally, the lab used databricks/dolly‑v2‑3b, but this model was removed from Hugging Face, causing 401/404 errors.

We resolved this by switching to a modern, lightweight, open model such as:

microsoft/phi‑2

TinyLlama‑1.1B‑Chat

These models are small enough to run on CPU and fully compatible with the lab.

#### 3. Loading the Model and Tokenizer
We loaded the model and tokenizer using:

python
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)
This step initializes the LLM and prepares it for inference.

#### 4. Creating the Text‑Generation Pipeline
We built a text‑generation pipeline with:

a maximum of 256 new tokens

device_map="auto" to automatically choose CPU/GPU

the loaded model and tokenizer

This pipeline is the engine that generates answers.

#### 5. Building the Vector Database Query
We used ChromaDB to:

Embed documents

Store them in a vector database

Retrieve the most relevant chunks using similarity search

The result of the query is a list of documents that serve as context for the RAG prompt.

#### 6. Constructing the Extended Prompt
We combined:

the retrieved context

the user’s question

into a single prompt:

python
Relevant context: {context}

The user's question: {question}
We also limited the context length to avoid memory issues:

python
context = context[:4096]
#### 7. Improving the Prompt with RAG Enforcement
To prevent hallucinations, we upgraded the prompt:

python
You are an assistant that answers ONLY using the provided context.
If the answer is not in the context, reply:
"The context does not contain the answer."
This forces the model to rely strictly on retrieved information.

#### 8. Testing the RAG System
Initially, we tested with an irrelevant question:

“Can I buy a Toshiba laptop?”

This exposed a key insight:

The model ignored the context and hallucinated an answer.

This happens because the question was unrelated to the dataset.

We corrected this by generating 10 questions that actually appear in the dataset, such as:

“What is BioAutoMATED?”

“Who won the Envisioning the Future of Computing Prize?”

“How does LiGO accelerate model training?”

These questions allowed the RAG system to perform correctly.

#### 9. Looping Through 10 Dataset‑Relevant Questions
We created a loop that:

builds a prompt for each question

feeds it into the pipeline

prints the model’s answer

This validated the RAG pipeline across multiple queries.

#### 10. Key Lessons Learned
✔ 1. RAG only works with relevant questions
If the question is unrelated to the dataset, the model will hallucinate.

✔ 2. Prompt engineering matters
A strict prompt dramatically reduces hallucinations.

✔ 3. Context length must be controlled
Large documents can overwhelm small models.

✔ 4. Small models behave differently than GPT‑4
They require stronger instructions and cleaner prompts.

✔ 5. Dolly‑v2 is no longer available
We learned how to adapt and replace deprecated models.

✔ 6. A working RAG pipeline requires all components
embeddings

vector search

prompt construction

LLM generation

All must work together.

#### 11. Final Conclusion
In this lab, we built a complete Retrieval‑Augmented Generation system from scratch using:

Hugging Face Transformers

A local LLM

ChromaDB for vector search

Custom prompt engineering

We learned how to:

load and configure models

build a text‑generation pipeline

retrieve relevant context

construct effective prompts

enforce grounding to avoid hallucinations

test the system with real dataset‑aligned questions

By the end, we had a fully functional RAG pipeline capable of answering questions using only the information stored in the vector database.